# LangChain vs LlamaIndex: A Detailed Comparison

Run top to bottom with a locally configured OpenAI key. Calls incur charges. The shared sales and feedback inputs are explicitly fictional; writes affect only an in-memory dictionary. Dependencies used in validation:

```bash
python -m pip install langchain-core==1.4.8 langchain-openai==1.1.11 openai==2.43.0 llama-index-core==0.14.25 llama-index-llms-openai==0.8.3 llama-index-embeddings-openai==0.7.0 numpy==2.5.4 certifi==2026.10.23
```

In [ ]:
import os, sys
from importlib.metadata import version
import numpy as np
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY locally before running. Never put it in a notebook cell.')
MODEL = os.getenv('MODEL', 'gpt-5.4-mini')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small')
print('Python:', sys.version.split()[0])
print('Chat model:', MODEL, 'Embedding model:', EMBEDDING_MODEL)
print('LangChain core:', version('langchain-core'), 'LlamaIndex core:', version('llama-index-core'))
# Fictional input shared by both frameworks, not a live business database.
records = [
    ('sales-feb', 'February 2025 sales revenue was 120000 USD.'),
    ('sales-mar', 'March 2025 sales revenue was 150000 USD.'),
    ('feedback-mar', 'March 2025 customer feedback: customers liked fast delivery. Satisfaction score: 4.2 out of 5.'),
]
print('Shared fictional source records:', len(records))

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = ChatOpenAI(model=MODEL, temperature=0, reasoning_effort='none',
                 max_tokens=512, max_retries=0, timeout=60)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, max_retries=0)
documents = [Document(page_content=text, metadata={'source': key}) for key, text in records]
store = InMemoryVectorStore(embeddings)
store.add_documents(documents, ids=[key for key, _ in records])
retriever = store.as_retriever(search_kwargs={'k': 2})
def retrieve_context(inputs):
    return '\n'.join(doc.page_content for doc in retriever.invoke(inputs['question']))
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer using only this fictional context and conversation. If absent, say unknown. Context: {context}'),
    MessagesPlaceholder('history'), ('human', '{question}')])
rag = RunnablePassthrough.assign(context=RunnableLambda(retrieve_context)) | prompt | llm
histories = {}
def get_history(session_id):
    if session_id not in histories:
        histories[session_id] = InMemoryChatMessageHistory()
    return histories[session_id]
def ask(question, session_id):
    history = get_history(session_id)
    response = rag.invoke({'question': question, 'history': history.messages})
    history.add_user_message(question)
    history.add_ai_message(response)
    return response
first = ask('What was March 2025 sales revenue in USD?', 'sales-demo')
followup = ask('What month and year did I just ask about?', 'sales-demo')
print(first.content)
print(followup.content)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
sequence = (
    RunnablePassthrough.assign(context=RunnableLambda(retrieve_context))
    .assign(answer=prompt | llm | StrOutputParser())
    .assign(result=RunnableLambda(lambda inputs: 'Processed result: ' + inputs['answer']))
)
processed = sequence.invoke({'question': 'Compare February and March 2025 sales. Give both amounts and the increase.', 'history': []})
print(processed['result'])
print('Retained named fields:', ', '.join(sorted(processed)))

In [ ]:
from llama_index.core import Document as LlamaDocument, VectorStoreIndex
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

llama_llm = LlamaOpenAI(model=MODEL, temperature=0, max_tokens=512,
    reasoning_effort='none', max_retries=0, timeout=60)
llama_embeddings = OpenAIEmbedding(model=EMBEDDING_MODEL, max_retries=0)
llama_docs = [LlamaDocument(text=text, id_=key, metadata={'source': key}) for key, text in records]
index = VectorStoreIndex.from_documents(llama_docs, embed_model=llama_embeddings)
query_engine = index.as_query_engine(llm=llama_llm, similarity_top_k=2)
sales_response = query_engine.query('What was March 2025 sales revenue in USD?')
print(str(sales_response))
print('Retrieved sources:', ', '.join(sorted(n.node.metadata['source'] for n in sales_response.source_nodes)))

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
# Only this in-memory teaching state can be changed by the agent.
feedback_state = {'score': 4.2, 'feedback': 'March 2025 customers liked fast delivery.'}
tool_events = []
def fetch_feedback(month: str) -> str:
    """Read fictional March 2025 customer feedback. Other months are unavailable."""
    if month != '2025-03':
        raise ValueError('Only 2025-03 exists in this teaching fixture')
    tool_events.append(('read', month))
    return feedback_state['feedback']
def update_score(score: float) -> str:
    """Set the fictional local satisfaction score, which must be between 0 and 5."""
    if not 0 <= score <= 5:
        raise ValueError('Score must be between 0 and 5')
    feedback_state['score'] = score
    tool_events.append(('write', score))
    return f'Local satisfaction score updated to {score} out of 5.'
agent = FunctionAgent(llm=llama_llm, tools=[fetch_feedback, update_score], streaming=False,
    system_prompt='Use the provided tools for every read or update. Do not invent feedback or claim a write without its tool result.')
read_result = await agent.run(user_msg='Fetch customer feedback for March 2025 using month 2025-03.', max_iterations=5)
write_result = await agent.run(user_msg='Update the local customer satisfaction score to 4.5 and report the saved score.', max_iterations=5)
print(str(read_result))
print(str(write_result))
print('Actual local tool events:', tool_events)

In [ ]:
import re
def compact(text):
    return re.sub(r'[^a-z0-9.]', '', str(text).lower())
assert '150000' in compact(first.content)
assert 'march' in followup.content.lower() and '2025' in followup.content
assert len(histories['sales-demo'].messages) == 4
assert get_history('isolated-session').messages == []
assert {'question', 'history', 'context', 'answer', 'result'} == set(processed)
assert all(value in compact(processed['answer']) for value in ['120000', '150000', '30000'])
assert processed['result'] == 'Processed result: ' + processed['answer']
assert '150000' in compact(sales_response)
assert 'sales-mar' in {node.node.metadata['source'] for node in sales_response.source_nodes}
assert feedback_state['score'] == 4.5 and ('read', '2025-03') in tool_events and ('write', 4.5) in tool_events
assert 'fast delivery' in str(read_result).lower() and '4.5' in str(write_result)
before = dict(feedback_state); previous_events = list(tool_events)
for invalid in [-0.1, 5.1, float('nan')]:
    try: update_score(invalid)
    except ValueError: pass
    else: raise AssertionError('Invalid score accepted')
try: fetch_feedback('1900-01')
except ValueError: pass
else: raise AssertionError('Unknown month accepted')
assert feedback_state == before and tool_events == previous_events
# Independently verify LangChain cosine ranking using the stored actual vectors.
query = 'What was March 2025 sales revenue in USD?'
q = np.asarray(embeddings.embed_query(query))
stored = list(store.store.values())
scores = [np.dot(q, d['vector']) / (np.linalg.norm(q) * np.linalg.norm(d['vector'])) for d in stored]
expected = [stored[i]['text'] for i in np.argsort(scores)[::-1][:2]]
assert [d.page_content for d in retriever.invoke(query)] == expected
assert sorted(key for key, _ in records) == sorted(d.metadata['source'] for d in documents)
print('PASS: grounded answers, 30000 USD increase, history isolation, named-key flow, actual read/write tools, invalid-input atomicity and independent cosine retrieval.')

## Original source-code appendix

These four archive blocks are retained verbatim as nonexecuting historical source. Use the tested implementations above.

```python
from langchain.llms import OpenAI
from langchain.retrievers import SimpleRetriever
from langchain.chains import SequentialChain
from langchain.memory import Memory

# Initialize the language model
llm = OpenAI(api_key="your_openai_api_key")
# Define a simple retriever
retriever = SimpleRetriever(source="https://example.com/data")
# Define memory for maintaining context
memory = Memory()
# Define a chain for processing the retrieved data
chain = SequentialChain(llm=llm, retriever=retriever, memory=memory)
# Execute the chain
response = chain.run("Retrieve and process data")
print(response)
```

```python
from langchain.chains import ComplexChain

# Define individual steps
def step1(input_text):
    return retriever.retrieve(input_text)
def step2(retrieved_data):
    return llm.generate(retrieved_data)
def step3(processed_data):
    return f"Processed result: {processed_data}"
# Create a complex chain
complex_chain = ComplexChain(steps=[step1, step2, step3], memory=memory)
# Execute the complex chain
result = complex_chain.run("Input text for processing")
print(result)
```

```python
from llamadb.connectors import DataConnector
from llamadb.engines import QueryEngine
from llamadb.llms import LLMWrapper

# Initialize a data connector
data_connector = DataConnector(source_type="database", connection_string="your_connection_string")
# Initialize the language model wrapper
llm_wrapper = LLMWrapper(model_name="gpt-3", api_key="your_openai_api_key")
# Initialize a query engine
query_engine = QueryEngine(data_connector=data_connector, llm_wrapper=llm_wrapper)
# Execute a natural language query
query = "Retrieve and analyze the latest sales data"
response = query_engine.query(query)
print(response)
```

```python
from llamadb.agents import DataAgent

# Initialize a data agent
data_agent = DataAgent(data_connector=data_connector, llm_wrapper=llm_wrapper)
# Perform a read operation
read_query = "Fetch customer feedback from the last month"
read_response = data_agent.read(read_query)
print(read_response)
# Perform a write operation
write_query = "Update the customer satisfaction score based on new feedback"
write_response = data_agent.write(write_query)
print(write_response)
```

## Original comparison table

The source table is retained here unchanged; the article table describes the runnable examples.

![Original comparison table](https://cdn-images-1.medium.com/max/800/1*bppvXHErC3GCdRe1tMl6Hg.png)